In [3]:
import pandas as pd
import os

raw_path = os.path.join("..", "Data", "Raw")
processed_path = os.path.join("..", "Data", "Processed")

folder_path = raw_path  # kept for consistency with earlier steps
os.listdir(folder_path)

['online_retail_II.csv']

In [4]:
file_path = os.path.join(folder_path, "online_retail_II.csv")

df = pd.read_csv(file_path)

df.shape

(1067371, 8)

In [5]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 65.1 MB


In [7]:
# Check how cancelled orders are marked - Invoice starting with 'C'
cancelled = df[df['Invoice'].astype(str).str.startswith('C')]
cancelled.shape

(19494, 8)

In [8]:
df_clean = df.dropna(subset=['Customer ID']).copy()
df_clean.shape

(824364, 8)

In [9]:
df_clean['Is_Cancelled'] = df_clean['Invoice'].astype(str).str.startswith('C')
df_clean['Is_Cancelled'].sum()

np.int64(18744)

In [10]:
cancellation_counts = df_clean.groupby('Customer ID')['Is_Cancelled'].sum().rename('Cancel_Count')
cancellation_counts.head()

Customer ID
12346.0    14
12347.0     0
12348.0     0
12349.0     5
12350.0     0
Name: Cancel_Count, dtype: int64

In [11]:
transactions = df_clean[~df_clean['Is_Cancelled']].copy()
transactions.shape

(805620, 9)

In [12]:
transactions['InvoiceDate'] = pd.to_datetime(transactions['InvoiceDate'])
transactions['InvoiceDate'].dtype

dtype('<M8[us]')

In [13]:
transactions[transactions['Quantity'] < 0].shape

(0, 9)

In [14]:
transactions[transactions['Price'] <= 0].shape

(71, 9)

In [15]:
transactions[transactions['Price'] <= 0]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Is_Cancelled
4674,489825,22076,6 RIBBONS EMPIRE,12,2009-12-02 13:34:00,0.0,16126.0,United Kingdom,False
6781,489998,48185,DOOR MAT FAIRY CAKE,2,2009-12-03 11:19:00,0.0,15658.0,United Kingdom,False
16107,490727,M,Manual,1,2009-12-07 16:38:00,0.0,17231.0,United Kingdom,False
18738,490961,22065,CHRISTMAS PUDDING TRINKET POT,1,2009-12-08 15:25:00,0.0,14108.0,United Kingdom,False
18739,490961,22142,CHRISTMAS CRAFT WHITE FAIRY,12,2009-12-08 15:25:00,0.0,14108.0,United Kingdom,False
...,...,...,...,...,...,...,...,...,...
1004540,577129,22464,HANGING METAL HEART LANTERN,4,2011-11-17 19:52:00,0.0,15602.0,United Kingdom,False
1005014,577168,M,Manual,1,2011-11-18 10:42:00,0.0,12603.0,Germany,False
1006110,577314,23407,SET OF 2 TRAYS HOME SWEET HOME,2,2011-11-18 13:23:00,0.0,12444.0,Norway,False
1011446,577696,M,Manual,1,2011-11-21 11:57:00,0.0,16406.0,United Kingdom,False


In [16]:
transactions = transactions[transactions['Price'] > 0].copy()
transactions.shape

(805549, 9)

In [17]:
transactions['TotalPrice'] = transactions['Quantity'] * transactions['Price']
transactions[['Quantity', 'Price', 'TotalPrice']].head()

,Quantity,Price,TotalPrice
0,12,6.95,83.4
1,12,6.75,81.0
2,12,6.75,81.0
3,48,2.10,100.8
4,24,1.25,30.0


In [18]:
transactions['Customer ID'] = transactions['Customer ID'].astype(int)
transactions['Customer ID'].dtype

dtype('int64')

In [19]:
transactions = transactions.merge(cancellation_counts, on='Customer ID', how='left')
transactions[['Customer ID', 'Cancel_Count']].head()

,Customer ID,Cancel_Count
0,13085,8
1,13085,8
2,13085,8
3,13085,8
4,13085,8


In [20]:
transactions.to_csv(os.path.join(processed_path, "transactions_clean.csv"), index=False)